# Inspect score meaning and preserve successive rankings

This offline example loads a saved DockingMT result, ranks it twice and inspects
the retained scalar evidence. All coordinates and scores are **invented software
fixtures**, not docking/rescoring predictions or scientific validation.

`example_affinity` is an empirical value in conventional kcal/mol;
`example_preference` is dimensionless. Neither is a measured physical energy or
calibrated probability. See [the contract](score_semantics.md) for actual Vina/Vinardo
descriptors and compatibility rules. The notebook requires the ordinary DockingMT
environment plus Jupyter; it performs no molecular operation or engine execution.

In [1]:
from pathlib import Path
import json
import sys

import pyunitwizard as puw
from dockingmt import DockingResult

directory = Path.cwd()
while not (directory / "devguide/validation/data/ranking/synthetic_result.json").is_file():
    if directory == directory.parent:
        raise FileNotFoundError("Run this notebook from the DockingMT checkout")
    directory = directory.parent
fixture = directory / "devguide/validation/data/ranking/synthetic_result.json"
saved = json.loads(fixture.read_text())
result = DockingResult.from_dict(saved)
assert result.to_dict() == saved
assert result.provenance["fixture_id"] == "ranking-example-v1"
print(result.provenance["evidence"])
print("Initial order:", [pose.pose_id for pose in result])
print("Optional providers loaded:", {name: name in sys.modules for name in ("vina", "molsysviewer")})
assert "vina" not in sys.modules and "molsysviewer" not in sys.modules

Synthetic software example; no engine, molecular preparation or scientific prediction
Initial order: ['A', 'B', 'C']
Optional providers loaded: {'vina': False, 'molsysviewer': False}


Descriptors accompany each pose's score numbers and declare their method,
version, interpretation, unit and comparison context. A missing descriptor means
unknown semantics. `score_definitions` returns an independent validated snapshot;
the definitions are stored in the existing metadata extension point.

In [2]:
for name, descriptor in result[0].score_definitions.items():
    print(name, "->", descriptor["kind"], descriptor["unit"], descriptor["preferred_direction"])
    print("  context:", descriptor["context"])
for pose in result:
    print(pose.pose_id, pose.scores)
assert result[0].scores["example_affinity"] == -8.0

example_affinity -> empirical kilocalorie / mole lower
  context: {'fixture_id': 'ranking-example-v1', 'purpose': 'invented numbers for software behavior only'}
example_preference -> dimensionless dimensionless higher
  context: {'fixture_id': 'ranking-example-v1', 'purpose': 'invented numbers for software behavior only'}
A {'example_affinity': -8.0, 'example_preference': 0.4}
B {'example_affinity': -6.0, 'example_preference': 0.9}
C {'example_affinity': -7.0, 'example_preference': 0.9}


Rank by the empirical example first, then by the dimensionless example.
Directions are explicit; the descriptor's preference is advisory. The two tied
dimensionless values preserve their order in the result supplied to the second
ranking. Neither operation modifies the input result or the earlier decision.

In [3]:
affinity_ranked = result.rank_by("example_affinity", ascending=True)
preference_ranked = affinity_ranked.rank_by("example_preference", ascending=False)
print("Empirical order:", [pose.pose_id for pose in affinity_ranked])
print("Dimensionless order:", [pose.pose_id for pose in preference_ranked])
assert [pose.pose_id for pose in affinity_ranked] == ["A", "C", "B"]
assert [pose.pose_id for pose in preference_ranked] == ["C", "B", "A"]
assert result.to_dict() == saved
assert len(affinity_ranked.ranking_history) == 1
assert len(preference_ranked.ranking_history) == 2
assert preference_ranked.provenance["ranking_policy"] == {"score_name": "example_preference", "ascending": False}

Empirical order: ['A', 'C', 'B']
Dimensionless order: ['C', 'B', 'A']


History retains the selected descriptor, observed values, input identities and
output indices. It records no coordinates. `output_order` refers to positions in
that entry's `input_order`, so repeated or missing IDs do not erase the local order.
This is inspection evidence, not an authenticated seal or a campaign identity.

In [4]:
history = preference_ranked.ranking_history
for number, entry in enumerate(history, start=1):
    print("Decision", number, entry["score_name"], "ascending:", entry["ascending"])
    print("  observations:", [(item["pose_id"], item["value"]) for item in entry["input_order"]])
    print("  output indices:", entry["output_order"])
assert history[0]["output_order"] == [0, 2, 1]
assert history[1]["output_order"] == [1, 2, 0]
assert all("coordinates" not in item for entry in history for item in entry["input_order"])

Decision 1 example_affinity ascending: True
  observations: [('A', -8.0), ('B', -6.0), ('C', -7.0)]
  output indices: [0, 2, 1]
Decision 2 example_preference ascending: False
  observations: [('A', 0.4), ('C', 0.9), ('B', 0.9)]
  output indices: [1, 2, 0]


Serialize and restore under a different session unit policy. Score numbers
keep their declared conventional unit; coordinate records keep their explicit nm
unit. Earlier observations remain independent of subsequent score edits and
exported snapshots.

In [5]:
exported = json.loads(json.dumps(preference_ranked.to_dict(), allow_nan=False))
with puw.context(standard_units=["angstrom", "fs", "kJ/mol"]):
    restored = DockingResult.from_dict(exported)
    assert restored.to_dict() == exported
    assert all(pose.to_dict()["coordinates"]["unit"] == "nanometer" for pose in restored)
before = {pose.pose_id: pose.to_dict() for pose in result}
for pose in restored:
    record = pose.to_dict()
    assert record["scores"] == before[pose.pose_id]["scores"]
    assert record["coordinates"] == before[pose.pose_id]["coordinates"]
    assert record["partner_state_id"] == before[pose.pose_id]["partner_state_id"]
    assert record["receptor_state_id"] == before[pose.pose_id]["receptor_state_id"]
restored[0].scores["example_affinity"] = 100.0
assert restored.ranking_history[0]["input_order"][0]["value"] == -8.0
exported["provenance"]["ranking_history"][0]["input_order"][0]["value"] = 999.0
assert preference_ranked.ranking_history[0]["input_order"][0]["value"] == -8.0
assert result.to_dict() == saved
print("Round trip, identity, units, input preservation and history independence: PASS")

Round trip, identity, units, input preservation and history independence: PASS


Fully undescribed legacy results remain explicitly rankable without inferred
meaning. A legacy latest policy is preserved as `legacy_policy_only`, with no
invented past score observations. Described rankings require matching descriptors
and state IDs; mixed, incompatible or malformed semantics fail before an output
is returned. See `tests/test_score_semantics.py` for those rejection guards.

Repeat this example with:

```bash
jupyter nbconvert --to notebook --execute --inplace devguide/validation/ranking_history.ipynb
```

Native Vina/Vinardo descriptor and initial-order checks are part of
`tests/test_engines.py::test_vina_backend_docking_execution`. Their minimal prepared
PDBQT controls validate the adapter boundary, not molecular chemistry or redocking.
The broader publication strategy remains [#29](https://github.com/uibcdf/dockingmt/issues/29).